In [3]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [4]:
train = pd.read_csv("../data/train_data.csv")
val = pd.read_csv("../data/val_data.csv")
test = pd.read_csv("../data/test_data.csv")

print(train.head())
print(train.columns)
print(train.shape, val.shape, test.shape)

     ID                                               Text  help  \
0  ID-2  Will it actually get better ??\n\n\n\nI (26F) ...     1   
1  ID-4  I can’t take it anymore\n\nI can’t take this s...     1   
2  ID-5  I struggle with self care in specific areas \n...     1   
3  ID-6  Keep going\n\nYou may not know how or when you...     0   
4  ID-7  Appointment for anxiety and depression - shoul...     1   

                                         Cause                  MH-condition  
0                                love failure   ['BPD', 'GAD', 'MDD', 'SUD']  
1                   trauma and unresolved pain                ['BPD', 'MDD']  
2                       executive dysfunction                        ['GAD']  
3  lack of support and mental health struggle                        ['MDD']  
4                       ineffective treatment   ['BPD', 'GAD', 'MDD', 'SUD']  
Index(['ID', 'Text', 'help', 'Cause', 'MH-condition'], dtype='str')
(1142, 5) (360, 5) (369, 5)


In [5]:
train = train[["Text", "help"]].dropna()
val = val[["Text", "help"]].dropna()
test = test[["Text", "help"]].dropna()

print(train["help"].value_counts())
print(val["help"].value_counts())
print(test["help"].value_counts())

help
1    648
0    494
Name: count, dtype: int64
help
1    191
0    169
Name: count, dtype: int64
help
1    210
0    159
Name: count, dtype: int64


In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_text"] = train["Text"].apply(clean_text)
val["clean_text"] = val["Text"].apply(clean_text)
test["clean_text"] = test["Text"].apply(clean_text)

In [7]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train = vectorizer.fit_transform(train["clean_text"])
X_val = vectorizer.transform(val["clean_text"])
X_test = vectorizer.transform(test["clean_text"])

y_train = train["help"]
y_val = val["help"]
y_test = test["help"]

In [8]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

val_pred_lr = log_reg.predict(X_val)
test_pred_lr = log_reg.predict(X_test)

print("Logistic Regression Validation Accuracy:", accuracy_score(y_val, val_pred_lr))
print("Logistic Regression Validation F1:", f1_score(y_val, val_pred_lr))

print("Logistic Regression Test Accuracy:", accuracy_score(y_test, test_pred_lr))
print("Logistic Regression Test F1:", f1_score(y_test, test_pred_lr))

Logistic Regression Validation Accuracy: 0.5944444444444444
Logistic Regression Validation F1: 0.6826086956521739
Logistic Regression Test Accuracy: 0.6070460704607046
Logistic Regression Test F1: 0.7151277013752456
